# QUANTUM LAB — Partícula en una caja 1D

Pozo de potencial infinito: estados ψₙ, energías Eₙ ∝ n², longitud de onda de De Broglie, nodos, energía de punto cero y superposición de estados dependiente del tiempo.

**Cómo usarlo en Google Colab**

1. Menú **Entorno de ejecución → Ejecutar todo** (o pulsa ▶ en cada celda, en orden). No hace falta instalar nada.
2. Al final aparece el simulador: los **parámetros** arriba y los resultados organizados en **pestañas**.
3. Escribe **cualquier valor** en las casillas y pulsa *Enter* (o mueve el deslizador, que amplía su rango solo).
   Se admiten formas como `2.5`, `2,5`, `1e-30`, `3*10^8` o `2*pi`.
4. Si un valor no tiene sentido físico (masa negativa, n = 2.5, una caja más pequeña que la longitud de Planck…),
   el simulador **no se detiene**: el apartado **«VALIDEZ FÍSICA DE LOS DATOS»** explica qué ocurre y por qué.
   - **[ERROR]**: con ese dato no se puede calcular (se explica el motivo físico).
   - **[AVISO]**: se calcula, pero el resultado no tiene respaldo físico (por ejemplo, el modelo deja de valer).
   - **[NOTA]**: información útil para interpretar el resultado.
5. Cada resultado se presenta como en los simuladores de Bohr y del efecto fotoeléctrico: parámetros, validez física,
   resultados alineados, tablas, interpretación física y gráficas.

Para ver el código de una celda, haz doble clic sobre su título.


In [ ]:
#@title 1) Utilidades comunes: formato de resultados, entradas sin límite y validez física { display-mode: "form" }
# =====================================================================================
#  Esta celda no contiene física: solo da formato a los resultados (recuadros, columnas
#  alineadas, tablas), lee los valores escritos por el usuario y explica los datos que
#  no tienen sentido físico. Ejecútala antes que las demás.
# =====================================================================================
import ast
import math
import operator
import textwrap
import warnings
import html as _html

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output, HTML
from scipy import constants as const

warnings.filterwarnings("ignore", message=".*IPython.display.IFrame.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------------
# Constantes físicas (CODATA, vía scipy.constants)
# ------------------------------------------------------------------
HBAR = const.hbar                    # J·s
H_PLANCK = const.h                   # J·s
C_LUZ = const.c                      # m/s
M_E = const.m_e                      # kg
EV = const.e                         # J por eV
K_B = const.k                        # J/K
A0 = const.physical_constants["Bohr radius"][0]          # m
E_H = const.physical_constants["Hartree energy"][0]      # J
L_PLANCK = const.physical_constants["Planck length"][0]  # m
KT_AMBIENTE_EV = K_B * 300 / EV      # energía térmica a 300 K  ≈ 0.0259 eV
ME_C2_EV = M_E * C_LUZ ** 2 / EV     # energía en reposo del electrón ≈ 511 keV

# ------------------------------------------------------------------
# Formato numérico
# ------------------------------------------------------------------
ANCHO = 80


def num(x, cifras=6):
    """Número legible: decimal si es «normal», notación científica si es muy grande o muy pequeño."""
    if x is None:
        return "—"
    try:
        x = float(x)
    except (TypeError, ValueError):
        return str(x)
    if math.isnan(x):
        return "no definido"
    if math.isinf(x):
        return "infinito" if x > 0 else "−infinito"
    if x == 0:
        return "0"
    if 1e-3 <= abs(x) < 1e6:
        texto = f"{x:.{cifras}g}"
    else:
        texto = f"{x:.{max(cifras - 2, 2)}e}"
    return texto.replace("-", "−")


def num_log(log10_valor, cifras=4):
    """Escribe 10^(log10_valor) aunque el número no quepa en un float (por ejemplo 10^5000)."""
    if not np.isfinite(log10_valor):
        return "0" if log10_valor < 0 else "infinito"
    exponente = math.floor(log10_valor)
    mantisa = 10 ** (log10_valor - exponente)
    if mantisa >= 9.9999999:
        mantisa, exponente = 1.0, exponente + 1
    if -3 <= exponente < 6:
        return num(10 ** log10_valor, cifras + 2)
    return f"{mantisa:.{cifras}f}e{exponente:+03d}".replace("-", "−")


def entero(n):
    """Entero con separador de miles (espacio fino) para leer números enormes."""
    return f"{int(n):,}".replace(",", " ")


_ESCALAS_LONGITUD = [(1e-15, "fm"), (1e-12, "pm"), (1e-9, "nm"), (1e-6, "µm"), (1e-3, "mm"), (1.0, "m"), (1e3, "km")]


def escala_longitud(L_m):
    """Unidad cómoda para dibujar una longitud L (en metros): devuelve (factor, nombre)."""
    for factor, nombre in reversed(_ESCALAS_LONGITUD):
        if L_m >= 0.1 * factor:
            return factor, nombre
    return 1e-15, "fm"


# ------------------------------------------------------------------
# Impresión con el estilo de QUANTUM LAB (recuadros y columnas alineadas)
# ------------------------------------------------------------------
def titulo_caja(titulo):
    print()
    print("╔" + "═" * (ANCHO - 2) + "╗")
    print("║" + titulo.center(ANCHO - 2) + "║")
    print("╚" + "═" * (ANCHO - 2) + "╝")


def seccion(titulo):
    print()
    print("─" * ANCHO)
    print(titulo)
    print("─" * ANCHO)


def dato(etiqueta, valor, unidad="", ancho_etiqueta=40):
    texto = f"{valor} {unidad}".rstrip()
    print(f"   {etiqueta:<{ancho_etiqueta}}: {texto}")


def parrafo(texto, sangria=3):
    for linea in textwrap.wrap(texto, ANCHO - sangria):
        print(" " * sangria + linea)


def vinetas(lineas, sangria=3):
    for linea in lineas:
        partes = textwrap.wrap(linea, ANCHO - sangria - 2)
        for i, parte in enumerate(partes):
            print(" " * sangria + ("• " if i == 0 else "  ") + parte)


def tabla(encabezados, filas, sangria=5, alinear=None):
    """Tabla con bordes de caja; cada columna se ajusta a su contenido. alinear: «c» centrado, «l» izquierda."""
    filas = [[str(c) for c in f] for f in filas]
    anchos = [max(len(str(h)), *(len(f[i]) for f in filas)) + 4 if filas else len(str(h)) + 4
              for i, h in enumerate(encabezados)]
    margen = " " * sangria

    def linea(izq, cruce, der):
        return margen + izq + cruce.join("─" * a for a in anchos) + der

    print(linea("┌", "┬", "┐"))
    alinear = alinear or "c" * len(encabezados)

    def celda(c, a, modo):
        return (" " + c).ljust(a) if modo == "l" else c.center(a)

    print(margen + "│" + "│".join(str(h).center(a) for h, a in zip(encabezados, anchos)) + "│")
    print(linea("├", "┼", "┤"))
    for f in filas:
        print(margen + "│" + "│".join(celda(c, a, m) for c, a, m in zip(f, anchos, alinear)) + "│")
    print(linea("└", "┴", "┘"))


# ------------------------------------------------------------------
# Validez física de los datos
# ------------------------------------------------------------------
class Diagnostico:
    """
    Reúne los comentarios sobre los datos introducidos:
      ERROR  no se puede calcular (por ejemplo, una masa negativa o n = 2.5);
      AVISO  se calcula, pero el resultado no tiene respaldo físico o el modelo deja de valer;
      NOTA   información útil para interpretar los resultados.
    """

    def __init__(self):
        self.items = []

    def error(self, texto):
        self.items.append(("ERROR", texto))

    def aviso(self, texto):
        self.items.append(("AVISO", texto))

    def nota(self, texto):
        self.items.append(("NOTA", texto))

    @property
    def hay_error(self):
        return any(nivel == "ERROR" for nivel, _ in self.items)

    def imprimir(self):
        titulo_caja("VALIDEZ FÍSICA DE LOS DATOS")
        if not self.items:
            print("   Todos los valores están dentro del rango de validez del modelo.")
            return
        orden = {"ERROR": 0, "AVISO": 1, "NOTA": 2}
        for nivel, texto in sorted(self.items, key=lambda it: orden[it[0]]):
            etiqueta = f"   [{nivel}] "
            partes = textwrap.wrap(texto, ANCHO - len(etiqueta))
            for i, parte in enumerate(partes):
                print((etiqueta if i == 0 else " " * len(etiqueta)) + parte)
            print()
        if self.hay_error:
            print("   >> Con estos datos no se puede calcular. Corrige los valores marcados como ERROR.")


def revisar_masa(m_me, diag, nombre="La masa"):
    """Comprueba una masa expresada en masas del electrón. Devuelve la masa en kg o None."""
    if m_me is None:
        return None
    if m_me == 0:
        diag.error(f"{nombre} vale 0. Una partícula sin masa (como el fotón) no obedece la ecuación de "
                   "Schrödinger: siempre viaja a la velocidad de la luz y necesita una descripción relativista. "
                   "Además, todas las energías del modelo serían infinitas (E ∝ 1/m).")
        return None
    if m_me < 0:
        diag.error(f"{nombre} es negativa ({num(m_me)} mₑ). No existe materia con masa negativa: la energía "
                   "cinética p²/2m saldría negativa y los estados ligados no tendrían sentido.")
        return None
    if m_me < 1e-6:
        diag.aviso(f"{nombre} ({num(m_me)} mₑ = {num(m_me * ME_C2_EV)} eV/c²) es menor que la de cualquier "
                   "partícula con masa conocida (los neutrinos tienen menos de ~1e−6 mₑ). Se calcula, pero no "
                   "corresponde a ninguna partícula real.")
    elif m_me > 1e15:
        diag.nota(f"{nombre} ({num(m_me * M_E)} kg) es la de un objeto macroscópico o casi. Su longitud de onda "
                  "de De Broglie es diminuta y los efectos cuánticos son imperceptibles: se comporta de forma "
                  "clásica (principio de correspondencia).")
    return m_me * M_E


def revisar_longitud(L_m, diag, nombre, que="longitud"):
    """Comentarios sobre una longitud física (en metros) ya validada como positiva."""
    if L_m < L_PLANCK:
        diag.aviso(f"{nombre} ({num(L_m)} m) es menor que la longitud de Planck (1.6e−35 m). A esa escala la "
                   "gravedad cuántica es importante y ninguna teoría física conocida es fiable: el resultado es "
                   "solo la evaluación matemática de la fórmula.")
    elif L_m < 1e-15:
        diag.aviso(f"{nombre} ({num(L_m)} m) es menor que un protón (≈ 1e−15 m). La partícula estaría confinada "
                   "en una región más pequeña que un núcleo: allí dominan la física nuclear y la relativista.")
    elif L_m > 1.0:
        diag.nota(f"{nombre} ({num(L_m)} m) es de tamaño macroscópico. Los niveles de energía quedan tan juntos que "
                  "el espectro es prácticamente continuo: es el límite clásico.")


def revisar_relatividad(log10_E_J, m_kg, diag, que="La energía"):
    """Compara una energía con la energía en reposo mc² (en escala logarítmica para no desbordar)."""
    log10_mc2 = math.log10(m_kg * C_LUZ ** 2)
    cociente = log10_E_J - log10_mc2
    if cociente > 0:
        diag.aviso(f"{que} es {num_log(cociente, 3)} veces la energía en reposo mc² de la partícula. Con tanta "
                   "energía se crearían pares partícula–antipartícula y la ecuación de Schrödinger (no relativista) "
                   "deja de valer: los números son el resultado matemático de la fórmula, no una predicción física.")
    elif cociente > -2:
        diag.aviso(f"{que} es el {num(100 * 10 ** cociente, 3)} % de mc²: la velocidad de la partícula es una "
                   "fracción apreciable de la de la luz y el modelo no relativista empieza a fallar.")


# ------------------------------------------------------------------
# Lectura segura de los valores escritos (admite 1e-30, 2,5, 2*pi, 3×10^8…)
# ------------------------------------------------------------------
_OPERACIONES = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
                ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
_NOMBRES = {"pi": math.pi, "e": math.e, "inf": math.inf, "infinito": math.inf}


def _evaluar(nodo):
    if isinstance(nodo, ast.Expression):
        return _evaluar(nodo.body)
    if isinstance(nodo, ast.Constant) and isinstance(nodo.value, (int, float)):
        return float(nodo.value)
    if isinstance(nodo, ast.Name) and nodo.id.lower() in _NOMBRES:
        return _NOMBRES[nodo.id.lower()]
    if isinstance(nodo, ast.BinOp) and type(nodo.op) in _OPERACIONES:
        a, b = _evaluar(nodo.left), _evaluar(nodo.right)
        if isinstance(nodo.op, ast.Pow) and abs(b) > 1e4 and abs(a) > 1:
            return math.inf if b > 0 else 0.0
        return _OPERACIONES[type(nodo.op)](a, b)
    if isinstance(nodo, ast.UnaryOp) and type(nodo.op) in _OPERACIONES:
        return _OPERACIONES[type(nodo.op)](_evaluar(nodo.operand))
    raise ValueError


def leer_numero(texto):
    """Convierte el texto escrito en un número. Devuelve (valor, None) o (None, explicación)."""
    t = str(texto).strip()
    if t == "":
        return None, "la casilla está vacía"
    t = (t.replace("π", "pi").replace("−", "-").replace("×", "*").replace("^", "**").replace(" ", ""))
    if t.count(",") == 1 and "." not in t:
        t = t.replace(",", ".")                          # coma decimal: 2,5 → 2.5
    t = t.replace("x10", "*10").replace("X10", "*10")
    try:
        valor = _evaluar(ast.parse(t, mode="eval"))
    except OverflowError:
        return None, (f"«{texto}» es mayor que 1.8e308, el número más grande que puede manejar el ordenador "
                      "(un float de 64 bits)")
    except ZeroDivisionError:
        return None, f"«{texto}» contiene una división entre cero"
    except (ValueError, SyntaxError, TypeError, RecursionError, MemoryError):
        return None, f"«{texto}» no es un número (ejemplos válidos: 2.5, 2,5, 1e-30, 3*10^8, 2*pi)"
    if isinstance(valor, complex):
        return None, f"«{texto}» da un número complejo"
    if math.isinf(valor) and "inf" not in t.lower():
        return None, (f"«{texto}» es mayor que 1.8e308, el número más grande que puede manejar el ordenador "
                      "(un float de 64 bits)")
    return float(valor), None


# ------------------------------------------------------------------
# Entradas: deslizador + casilla de texto, SIN límite de rango
# ------------------------------------------------------------------
_ESTILO = {"description_width": "0px"}


class Entrada:
    """
    Control con deslizador y casilla de texto enlazados. La casilla admite CUALQUIER valor: si se escribe
    uno fuera del rango del deslizador, el deslizador amplía su rango. El valor que se usa es el de la casilla.
    tipo: "real", "entero" o "log" (deslizador logarítmico, para masas, longitudes o frecuencias).
    """

    def __init__(self, etiqueta, valor, unidad="", tipo="real", rango=(0.0, 1.0), paso=None, ayuda=""):
        self.etiqueta, self.tipo = etiqueta, tipo
        self._sincronizando = False
        disposicion = w.Layout(width="290px")
        if tipo == "entero":
            self.deslizador = w.IntSlider(value=int(valor), min=int(rango[0]), max=int(rango[1]), step=1,
                                          readout=False, continuous_update=False, layout=disposicion, style=_ESTILO)
        elif tipo == "log":
            self.deslizador = w.FloatLogSlider(value=float(valor), base=10, min=rango[0], max=rango[1],
                                               step=paso or 0.05, readout=False, continuous_update=False,
                                               layout=disposicion, style=_ESTILO)
        else:
            self.deslizador = w.FloatSlider(value=float(valor), min=rango[0], max=rango[1],
                                            step=paso or (rango[1] - rango[0]) / 200, readout=False,
                                            continuous_update=False, layout=disposicion, style=_ESTILO)
        self._rango_original = (self.deslizador.min, self.deslizador.max)
        self.texto = w.Text(value=self._formato(valor), continuous_update=False, layout=w.Layout(width="150px"))
        etiqueta_html = w.HTML(f"<div style='padding-top:4px' title='{_html.escape(ayuda)}'>{_html.escape(etiqueta)}</div>",
                               layout=w.Layout(width="250px"))
        unidad_html = w.HTML(f"<div style='padding-top:4px;opacity:.75'>{_html.escape(unidad)}</div>",
                             layout=w.Layout(width="200px"))
        self.caja = w.HBox([etiqueta_html, self.deslizador, self.texto, unidad_html],
                           layout=w.Layout(align_items="center", margin="1px 0"))
        self.deslizador.observe(self._desde_deslizador, "value")
        self.texto.observe(self._desde_texto, "value")

    def _formato(self, v):
        if self.tipo == "entero":
            return str(int(round(v)))
        return num(v, 6).replace("−", "-")

    def _desde_deslizador(self, cambio):
        if self._sincronizando:
            return
        self._sincronizando = True
        self.texto.value = self._formato(cambio["new"])
        self._sincronizando = False

    def _desde_texto(self, cambio):
        if self._sincronizando:
            return
        v, _ = leer_numero(cambio["new"])
        if v is None or not math.isfinite(v):
            return
        self._sincronizando = True
        try:
            d = self.deslizador
            lo, hi = self._rango_original
            # Si el valor vuelve a caber en el rango inicial, el deslizador recupera ese rango (más cómodo de usar)
            if self.tipo == "log" and v > 0 and lo <= math.log10(v) <= hi:
                d.min, d.max = lo, hi
            elif self.tipo != "log" and lo <= v <= hi:
                d.min, d.max = lo, hi
            if self.tipo == "log":
                if v > 0:
                    e = math.log10(v)
                    if e < d.min:
                        d.min = math.floor(e)
                    if e > d.max:
                        d.max = math.ceil(e)
                    d.value = v
            elif self.tipo == "entero":
                if float(v).is_integer() and abs(v) < 2 ** 31:
                    vi = int(v)
                    if vi < d.min:
                        d.min = vi
                    if vi > d.max:
                        d.max = vi
                    d.value = vi
            else:
                if v < d.min:
                    d.min = v
                if v > d.max:
                    d.max = v
                d.value = v
        except Exception:
            pass
        finally:
            self._sincronizando = False

    def leer(self, diag):
        """Valor numérico de la casilla; si no se puede leer, añade un ERROR al diagnóstico y devuelve None."""
        v, problema = leer_numero(self.texto.value)
        if problema:
            diag.error(f"{self.etiqueta}: {problema}.")
            return None
        if not math.isfinite(v):
            diag.error(f"{self.etiqueta} es infinito. Ninguna magnitud física medible es infinita; "
                       "escribe un valor finito, por grande que sea.")
            return None
        return v

    def fijar(self, v):
        self.texto.value = self._formato(v)

    def al_cambiar(self, funcion):
        self.texto.observe(lambda _: funcion(), "value")


def revisar_entero(v, etiqueta, diag, explicacion):
    """Comprueba que un número cuántico sea entero; si no lo es, explica por qué debe serlo."""
    if v is None:
        return None
    if abs(v - round(v)) > 1e-9 * max(1.0, abs(v)):
        diag.error(f"{etiqueta} = {num(v)} no es entero. {explicacion}")
        return None
    if abs(v) > 1e300:
        diag.error(f"{etiqueta} = {num(v)} es demasiado grande para representarlo (supera 1e300).")
        return None
    return int(round(v))


# ------------------------------------------------------------------
# Paneles de la interfaz
# ------------------------------------------------------------------
_BORDE = "1px solid rgba(128,128,128,0.45)"


def panel(titulo, filas):
    cabecera = w.HTML(f"<div style='font:600 11px monospace;letter-spacing:.14em;text-transform:uppercase;"
                      f"opacity:.7;padding-bottom:3px;border-bottom:{_BORDE};margin-bottom:4px'>"
                      f"{_html.escape(titulo)}</div>")
    return w.VBox([cabecera] + list(filas), layout=w.Layout(border=_BORDE, padding="8px 12px", margin="6px 0"))


def encabezado(titulo, subtitulo):
    display(HTML(f"<div style='margin:4px 0 10px 0'><div style='font:600 11px monospace;letter-spacing:.2em;"
                 f"opacity:.65'>QUANTUM LAB</div><div style='font-size:22px;font-weight:600;margin:2px 0'>"
                 f"{_html.escape(titulo)}</div><div style='opacity:.75'>{_html.escape(subtitulo)}</div></div>"))


def ayuda_entradas():
    return w.HTML("<div style='opacity:.7;font-size:12px;margin:2px 0 4px 0'>Escribe cualquier valor en las casillas "
                  "(admite 1e-30, 2,5, 3*10^8, 2*pi) y pulsa Enter, o usa el deslizador. Si un valor no tiene "
                  "sentido físico, el apartado «VALIDEZ FÍSICA DE LOS DATOS» explica qué ocurre.</div>")


def selector(etiqueta, opciones, valor=None, ancho="290px"):
    menu = w.Dropdown(options=opciones, value=valor, layout=w.Layout(width=ancho), style=_ESTILO)
    fila = w.HBox([w.HTML(f"<div style='padding-top:4px'>{_html.escape(etiqueta)}</div>", layout=w.Layout(width="250px")),
                   menu], layout=w.Layout(align_items="center", margin="1px 0"))
    return menu, fila


def casilla(texto, valor=False):
    return w.Checkbox(value=valor, description=texto, indent=False, layout=w.Layout(width="auto", margin="0 18px 0 0"))


def boton(texto, funcion, estilo="primary"):
    b = w.Button(description=texto, button_style=estilo, layout=w.Layout(width="auto", margin="4px 8px 4px 0"))
    b.on_click(lambda _: funcion())
    return b


def ejecutar_en(salida, funcion):
    """Ejecuta un cálculo dentro de una salida; cualquier fallo imprevisto se explica en lugar de romper el programa."""
    with salida:
        clear_output(wait=True)
        try:
            funcion()
        except Exception as error:      # noqa: BLE001 — se informa al usuario
            titulo_caja("NO SE PUDO COMPLETAR EL CÁLCULO")
            parrafo(f"Ocurrió un problema numérico inesperado: {type(error).__name__}: {error}")
            parrafo("Suele deberse a valores tan extremos que superan la precisión del ordenador. Prueba con "
                    "valores menos extremos; el resto del simulador sigue funcionando.")


# ------------------------------------------------------------------
# Estilo de las gráficas (fondo claro, como los simuladores de Bohr y del efecto fotoeléctrico)
# ------------------------------------------------------------------
COLOR = {"onda": "#2471a3", "densidad": "#d35400", "verde": "#147a4b", "rojo": "#c0392b", "morado": "#8e44ad",
         "naranja": "#f39c12", "gris": "#7f8c8d", "oscuro": "#34495e", "turquesa": "#16a085", "azul_claro": "#5dade2"}


def estilo_graficas():
    plt.rcdefaults()
    plt.rcParams.update({
        "figure.dpi": 100, "savefig.dpi": 110, "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
        "axes.labelsize": 11.5, "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.35,
        "legend.fontsize": 9.5, "legend.framealpha": 0.95, "axes.spines.top": False, "axes.spines.right": False,
    })


def recuadro(ax, texto, x=0.97, y=0.95, color="#f39c12", ha="right", va="top", fondo="#fef9e7", tam=9.5):
    """Recuadro informativo como los de las gráficas de Bohr y del efecto fotoeléctrico."""
    ax.text(x, y, texto, transform=ax.transAxes, ha=ha, va=va, fontsize=tam, linespacing=1.5, zorder=10,
            bbox=dict(boxstyle="round,pad=0.55", facecolor=fondo, edgecolor=color, linewidth=1.6, alpha=0.96))


def mostrar(fig):
    """Muestra una figura de Matplotlib o de Plotly dentro de la salida de un widget."""
    if hasattr(fig, "savefig"):
        display(fig)
        plt.close(fig)
    else:
        pagina = fig.to_html(include_plotlyjs="cdn", full_html=True, config={"displaylogo": False, "responsive": True})
        alto = int(fig.layout.height or 600) + 20
        display(HTML(f'<iframe srcdoc="{_html.escape(pagina, quote=True)}" '
                     f'style="width:100%;height:{alto}px;border:0;background:white"></iframe>'))


estilo_graficas()
print("Utilidades de QUANTUM LAB listas.")


In [ ]:
#@title 2) Física de la partícula en una caja 1D { display-mode: "form" }
# =====================================================================================
#  Pozo infinito de anchura a:  V = 0 dentro (0 < x < a), V = ∞ fuera.
#
#      ψₙ(x) = √(2/a)·sin(nπx/a)          Eₙ = n²π²ħ²/(2ma²)          n = 1, 2, 3, …
#
#  Superposición:  Ψ(x,t) = Σ cₖ ψₖ(x) e^{−iEₖt/ħ}.  Con τ = E₁t/ħ la fase de cada término es k²τ,
#  así que la evolución se repite exactamente cada Δτ = 2π.
#  Las magnitudes de la superposición (⟨x⟩, probabilidad en una región, ⟨E⟩) se calculan con fórmulas
#  CERRADAS: son exactas para cualquier n, incluso n = 10⁹, donde una malla numérica no alcanzaría.
# =====================================================================================

N_DIBUJO_MAX = 5000        # hasta este n se dibuja ψ en toda la caja; por encima, una ventana ampliada
N_MAPA_MAX = 200           # hasta este n se dibuja el mapa espacio–tiempo de la superposición


def log10_energia(n, a_m, m_kg):
    """log₁₀ de Eₙ en julios (en logaritmos para que n o a extremos no desborden)."""
    return 2 * math.log10(abs(n)) + math.log10(math.pi ** 2 * HBAR ** 2 / 2) - math.log10(m_kg) - 2 * math.log10(a_m)


def energia_nivel(n, a_m, m_kg):
    """Eₙ = n²π²ħ²/(2ma²)  [J]."""
    return n ** 2 * math.pi ** 2 * HBAR ** 2 / (2 * m_kg * a_m ** 2)


def funcion_onda(x, n, a):
    """ψₙ(x) = √(2/a)·sin(nπx/a)."""
    return np.sqrt(2 / a) * np.sin(n * np.pi * np.asarray(x) / a)


def integrar(x, y):
    return float((getattr(np, "trapezoid", None) or np.trapz)(y, x))


def incertidumbre_posicion(n, a):
    """Δx = a·√(1/12 − 1/(2n²π²))."""
    return a * math.sqrt(1 / 12 - 1 / (2 * n ** 2 * math.pi ** 2))


# ------------------------------------------------------------------
# Validación de los datos (con explicación física de cada problema)
# ------------------------------------------------------------------
EXPLICACION_N = ("En una caja la función de onda debe anularse en las dos paredes. sin(nπx/a) vale 0 en x = a "
                 "solo si n es entero: un n no entero no cumple la condición de contorno y no es un estado "
                 "permitido. Por eso la energía está cuantizada.")


def revisar_n(n_valor, diag, etiqueta="n"):
    n = revisar_entero(n_valor, etiqueta, diag, EXPLICACION_N)
    if n is None:
        return None, 1
    if n == 0:
        diag.error(f"{etiqueta} = 0 da ψ(x) = √(2/a)·sin(0) = 0 en toda la caja: la probabilidad total sería 0, es "
                   "decir, no habría partícula. El estado de menor energía es n = 1, y su energía NO es cero "
                   "(energía de punto cero, consecuencia del principio de incertidumbre).")
        return None, 1
    if n < 0:
        diag.aviso(f"{etiqueta} = {n} es negativo. ψ₋ₖ = −ψₖ: solo cambia el signo global de la función de onda (una "
                   f"fase que no se puede medir), así que describe el mismo estado físico que {etiqueta} = {-n}. "
                   f"Se usa {etiqueta} = {-n}.")
        return -n, -1
    return n, 1


def revisar_caja(a_nm, m_me, diag):
    """Valida la longitud y la masa. Devuelve (a [m], m [kg]) o (None, None)."""
    a_m = None
    if a_nm is not None:
        if a_nm == 0:
            diag.error("La longitud de la caja es 0: no hay espacio para la partícula. Las energías serían infinitas "
                       "(Eₙ ∝ 1/a²): cuanto más se confina una partícula, más energía cinética tiene.")
        elif a_nm < 0:
            diag.error(f"La longitud de la caja es negativa ({num(a_nm)} nm). Una longitud física no puede ser "
                       "negativa; las paredes estarían «al revés».")
        else:
            a_m = a_nm * 1e-9
    m_kg = revisar_masa(m_me, diag)
    if a_m is not None:
        revisar_longitud(a_m, diag, "La longitud de la caja")
    return a_m, m_kg


# ------------------------------------------------------------------
# Superposición: coeficientes y fórmulas cerradas
# ------------------------------------------------------------------
def coeficientes(estados, diag):
    """
    estados: lista de (n, amplitud, fase) ya leídos (pueden ser None).
    Devuelve (lista de n, coeficientes complejos normalizados) o (None, None) si hay un error.
    """
    combinados = {}
    for i, (n_v, A, fase) in enumerate(estados, 1):
        etiqueta = f"n del estado {i}"
        n, signo = revisar_n(n_v, diag, etiqueta)
        if n is None or A is None or fase is None:
            continue
        if A == 0:
            diag.nota(f"La amplitud del estado {i} es 0: ese estado no contribuye a la superposición.")
            continue
        if A < 0:
            diag.nota(f"La amplitud del estado {i} es negativa ({num(A)}). Equivale a amplitud {num(-A)} con la fase "
                      "aumentada en π (multiplicar por −1 = e^{iπ}).")
        if abs(fase) > 2 * math.pi:
            diag.nota(f"La fase del estado {i} ({num(fase)} rad) supera 2π. Las fases se repiten cada 2π: equivale a "
                      f"{num(math.remainder(fase, 2 * math.pi))} rad.")
        c = signo * A * complex(math.cos(fase), math.sin(fase))
        if n in combinados:
            diag.nota(f"El nivel n = {n} aparece más de una vez: sus coeficientes se suman (un mismo estado no se "
                      "puede contar dos veces en una superposición).")
        combinados[n] = combinados.get(n, 0) + c
    if diag.hay_error:
        return None, None
    combinados = {n: c for n, c in combinados.items() if abs(c) > 0}
    if not combinados:
        diag.error("Todas las amplitudes son cero (o los estados se anulan entre sí): la función de onda sería 0 en "
                   "toda la caja y no habría partícula. Activa al menos un estado con amplitud distinta de 0.")
        return None, None
    ns = np.array(sorted(combinados), dtype=float)
    cs = np.array([combinados[int(n)] for n in ns], dtype=complex)
    norma = math.sqrt(float(np.sum(np.abs(cs) ** 2)))
    return ns, cs / norma


def _seno_sobre(q, u):
    """∫₀ᵘ cos(qπs) ds = sin(qπu)/(qπ)   (u en fracciones de a)."""
    q = np.asarray(q, dtype=float)
    con_q = np.abs(q) > 0
    return np.where(con_q, np.sin(q * np.pi * u) / np.where(con_q, q * np.pi, 1.0), u)


def matriz_region(ns, u1, u2):
    """S_jk = ∫ ψⱼψₖ dx sobre [u1·a, u2·a]  (exacta, adimensional)."""
    J, K = np.meshgrid(ns, ns, indexing="ij")
    return (_seno_sobre(J - K, u2) - _seno_sobre(J - K, u1)) - (_seno_sobre(J + K, u2) - _seno_sobre(J + K, u1))


def matriz_posicion(ns):
    """X_jk = ∫ ψⱼ x ψₖ dx / a   (exacta): 1/2 en la diagonal y −8jk/(π²(j²−k²)²) si j + k es impar."""
    J, K = np.meshgrid(ns, ns, indexing="ij")
    X = np.zeros_like(J)
    impar = (np.mod(J + K, 2) == 1)
    X[impar] = -8 * J[impar] * K[impar] / (math.pi ** 2 * (J[impar] ** 2 - K[impar] ** 2) ** 2)
    X[J == K] = 0.5
    return X


def valor_esperado(ns, cs, matriz, tau):
    """Σ c*ⱼ cₖ e^{i(j²−k²)τ} M_jk  (real)."""
    fase = np.exp(-1j * (ns ** 2) * tau)
    b = cs * fase
    return float(np.real(np.conj(b) @ matriz @ b))


def onda_total(x_frac, ns, cs, tau, a):
    """Ψ(x, τ) en la malla x = x_frac·a."""
    psi = np.zeros_like(x_frac, dtype=complex)
    for n, c in zip(ns, cs):
        psi += c * np.exp(-1j * n ** 2 * tau) * np.sqrt(2 / a) * np.sin(n * np.pi * x_frac)
    return psi


In [ ]:
#@title 3) Gráficas de la partícula en una caja { display-mode: "form" }
def _paredes(ax, x0, x1):
    for xp in (x0, x1):
        ax.axvline(xp, color=COLOR["oscuro"], lw=3.2, zorder=4)


def figura_estado(n, a_m, m_kg, logE):
    """ψₙ, |ψₙ|² y diagrama de niveles. Para n muy grande se dibuja una ventana junto a la pared."""
    completo = n <= N_DIBUJO_MAX
    if completo:
        x = np.linspace(0, a_m, max(2001, 40 * n + 1))
        x_fin = a_m
    else:
        x_fin = 3 * a_m / n                       # tres semilongitudes de onda desde la pared x = 0
        x = np.linspace(0, x_fin, 1501)
    f, u = escala_longitud(x_fin)
    a_vis = a_m / f
    # Se usa la fase t = n·x/a para no perder precisión con n enormes
    t = np.linspace(0, n * x_fin / a_m, len(x))
    psi = math.sqrt(2 / a_m) * np.sin(np.pi * t)
    rho = psi ** 2
    xs = x / f
    amp_unidad = f"{u}" + "$^{-1/2}$"
    if not completo:
        # Ventana diminuta: se usan unidades naturales (x en semilongitudes de onda a/n, ψ en √(2/a))
        xs, f, u = t, a_m, "a"
        etiqueta_x = f"posición x / (a/n)      (a/n = {num(a_m / n, 3)} m)"
    else:
        etiqueta_x = f"posición x [{u}]"

    fig = plt.figure(figsize=(13.5, 13.0), constrained_layout=True)
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 0.9, 0.95])
    ax1, ax2 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
    axE, ax3 = fig.add_subplot(gs[1, :]), fig.add_subplot(gs[2, :])
    escala_psi = math.sqrt(f)                    # ψ en unidades de (unidad)^(−1/2)
    if not completo:
        amp_unidad = "a$^{-1/2}$"
    ax1.plot(xs, psi * escala_psi, color=COLOR["onda"], lw=2.4, label=fr"$\psi_{{{n}}}(x)$" if n < 1e6 else r"$\psi_n(x)$")
    ax1.fill_between(xs, psi * escala_psi, where=psi > 0, color=COLOR["onda"], alpha=0.15)
    ax1.fill_between(xs, psi * escala_psi, where=psi < 0, color=COLOR["rojo"], alpha=0.12)
    ax1.axhline(0, color="black", lw=0.9, alpha=0.7)
    if completo and 1 < n <= 60:
        nodos = np.arange(1, n) * a_m / n / f
        ax1.plot(nodos, np.zeros_like(nodos), "o", color=COLOR["rojo"], ms=6, zorder=5,
                 label=f"nodos interiores ({n - 1})")
    _paredes(ax1, 0, a_vis if completo else 0)
    ax1.set_xlabel(etiqueta_x)
    ax1.set_ylabel(f"ψ(x)  [{amp_unidad}]")
    ax1.set_title("Función de onda" + ("" if completo else " (ventana junto a la pared x = 0)"))
    ax1.legend(loc="upper right")

    ax2.plot(xs, rho * f, color=COLOR["densidad"], lw=2.4, label=fr"$|\psi(x)|^2$")
    ax2.fill_between(xs, rho * f, color=COLOR["densidad"], alpha=0.18)
    ax2.axhline(f / a_m, color=COLOR["gris"], ls="--", lw=1.5, label="promedio clásico 1/a")
    if completo and n <= 60:
        maximos = (2 * np.arange(1, n + 1) - 1) * a_m / (2 * n) / f
        ax2.plot(maximos, np.full_like(maximos, 2 * f / a_m), "*", color=COLOR["verde"], ms=11, zorder=5,
                 label=f"máximos de probabilidad ({n})")
    _paredes(ax2, 0, a_vis if completo else 0)
    ax2.set_xlabel(etiqueta_x)
    ax2.set_ylabel(f"|ψ(x)|²  [{u}$^{{-1}}$]")
    ax2.set_title("Densidad de probabilidad" + ("" if completo else " (ventana ampliada)"))
    ax2.set_ylim(0, 3.3 * f / a_m)
    ax2.legend(loc="upper right")
    maximos_txt = "1 máximo" if n == 1 else f"{entero(n)} máximos"
    nodos_txt = "1 nodo interior" if n == 2 else f"{entero(n - 1)} nodos interiores"
    texto = (f"n = {entero(n)}\n{nodos_txt}\n{maximos_txt}\n"
             f"λ = 2a/n = {num(2 * a_m / n / f, 4)} {u}" if completo else
             f"n = {entero(n)}\n{nodos_txt}\n{maximos_txt}\nλ = 2a/n = {num(2 * a_m / n, 4)} m")
    if not completo:
        texto += f"\nSe muestran 3 de las {entero(n)}\nsemilongitudes de onda"
    recuadro(ax2, texto, x=0.03, y=0.95, ha="left", color=COLOR["densidad"])

    # Energía frente a posición: V(x) = 0 dentro de la caja, V → ∞ en las paredes, nivel Eₙ y |ψ|² sobre el nivel
    log_Eev = logE - math.log10(EV)
    en_ev = math.isfinite(log_Eev) and -250 < log_Eev < 250
    E_nivel = 10 ** log_Eev if en_ev else 1.0
    unidad_E = "eV" if en_ev else "Eₙ"
    x_ini, x_fin_v = xs[0], xs[-1]
    tope = 1.9 * E_nivel
    escala_rho = 0.55 * E_nivel / max(float(rho.max()), 1e-300)
    axE.plot([x_ini, x_fin_v], [0, 0], color=COLOR["morado"], lw=4, solid_capstyle="butt", zorder=4,
             label="V = 0 dentro de la caja")
    axE.vlines([x_ini] + ([x_fin_v] if completo else []), 0, tope, color=COLOR["morado"], lw=4, zorder=4)
    axE.axhline(E_nivel, color=COLOR["rojo"], ls="--", lw=2.2, label=f"nivel Eₙ = {num_log(log_Eev)} eV")
    axE.plot(xs, E_nivel + rho * escala_rho, color=COLOR["densidad"], lw=2.2, label="|ψₙ|² sobre su nivel (escala arbitraria)")
    axE.fill_between(xs, E_nivel, E_nivel + rho * escala_rho, color=COLOR["densidad"], alpha=0.15)
    axE.text(x_ini, tope * 0.97, " V → ∞", color=COLOR["morado"], va="top", fontsize=10, fontweight="bold")
    if completo:
        axE.text(x_fin_v, tope * 0.97, "V → ∞ ", color=COLOR["morado"], va="top", ha="right", fontsize=10, fontweight="bold")
    axE.set_ylim(-0.08 * tope, tope)
    axE.set_xlabel(etiqueta_x)
    axE.set_ylabel(f"Energía [{unidad_E}]")
    axE.set_title("Energía frente a posición: potencial V(x) y nivel de energía Eₙ")
    axE.legend(loc="lower center", bbox_to_anchor=(0.5, 0.06), ncol=3, fontsize=9)

    # Diagrama de niveles (energías relativas reales, en unidades de E₁)
    if n <= 6:
        ks = np.arange(1, 7)
    else:
        ks = np.arange(max(1, n - 3), n + 4, dtype=float) if n < 1e15 else np.array([n])
    for k in ks:
        sel = (k == n)
        y = (k / n) ** 2 if n > 6 else k ** 2
        ax3.hlines(y, 1.0, 5.0, color=COLOR["rojo"] if sel else COLOR["onda"], lw=3.5 if sel else 2.2, zorder=3)
        ax3.text(0.85, y, f"n = {entero(k)}", ha="right", va="center", fontsize=11, fontweight="bold" if sel else "normal")
        if n <= 6:
            etiqueta = f"{num_log(logE - math.log10(EV) + 2 * math.log10(k / n))} eV"
        elif k == n:
            etiqueta = "Eₙ"
        else:
            # Eₖ − Eₙ = (k − n)(k + n)·E₁: se escribe la diferencia, que es lo que distingue niveles tan altos
            log_dif = math.log10(abs(k - n) * (k + n)) + logE - 2 * math.log10(n) - math.log10(EV)
            etiqueta = f"Eₙ {'+' if k > n else '−'} {num_log(log_dif)} eV"
        ax3.text(5.15, y, etiqueta, ha="left", va="center", fontsize=10.5)
    y_n = 1 if n > 6 else n ** 2
    info = (f"ESTADO n = {entero(n)}\n\nEₙ = {num_log(logE - math.log10(EV))} eV\n"
            f"Eₙ = {num_log(logE)} J\nEₙ / E₁ = n² = {num_log(2 * math.log10(n))}\n"
            f"Eₙ₊₁ − Eₙ = (2n+1)E₁ = {num_log(math.log10(2 * n + 1) + logE - 2 * math.log10(n) - math.log10(EV))} eV")
    ax3.text(7.0, y_n, info, ha="left", va="center",
             fontsize=10.5, linespacing=1.6,
             bbox=dict(boxstyle="round,pad=0.7", facecolor="#fef9e7", edgecolor=COLOR["rojo"], linewidth=2))
    ax3.set_xlim(0, 11)
    ys = [(k / n) ** 2 if n > 6 else k ** 2 for k in ks]
    margen = 0.12 * (max(ys) - min(ys) or 1)
    ax3.set_ylim(min(ys) - margen, max(ys) + margen)
    ax3.set_xticks([])
    ax3.set_yticks([])
    for lado in ("top", "right", "left", "bottom"):
        ax3.spines[lado].set_visible(False)
    ax3.grid(False)
    ax3.set_title("Niveles de energía Eₙ = n²·E₁ (separación a escala; crece con n)")
    fig.suptitle(f"QUANTUM LAB — Partícula en una caja 1D · estado n = {entero(n)}", fontsize=16, fontweight="bold")
    return fig


def figura_superposicion(ns, cs, tau, a_m, u1, u2, individuales):
    """Ψ(x, τ), |Ψ|² con la región, mapa espacio–tiempo y evolución de ⟨x⟩ y de la probabilidad."""
    f, u = escala_longitud(a_m)
    representable = ns.max() <= N_DIBUJO_MAX
    mapa = ns.max() <= N_MAPA_MAX
    taus = np.linspace(0, 2 * np.pi, 361)
    S = matriz_region(ns, u1, u2)
    X = matriz_posicion(ns)
    P_t = np.array([valor_esperado(ns, cs, S, t) for t in taus])
    x_t = np.array([valor_esperado(ns, cs, X, t) for t in taus])

    filas = 2 if representable else 1
    fig = plt.figure(figsize=(13.5, 9.6 if representable else 4.8), constrained_layout=True)
    gs = fig.add_gridspec(filas, 2)
    if representable:
        puntos = max(1201, 30 * int(ns.max()) + 1)
        xf = np.linspace(0, 1, puntos)
        psi = onda_total(xf, ns, cs, tau, a_m)
        rho = np.abs(psi) ** 2
        xs = xf * a_m / f
        ax1, ax2 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
        esc = math.sqrt(f)
        ax1.plot(xs, psi.real * esc, color=COLOR["onda"], lw=2.2, label="Re Ψ")
        ax1.plot(xs, psi.imag * esc, color=COLOR["morado"], lw=2.0, ls="--", label="Im Ψ")
        ax1.axhline(0, color="black", lw=0.8, alpha=0.7)
        _paredes(ax1, 0, a_m / f)
        ax1.set_xlabel(f"posición x [{u}]")
        ax1.set_ylabel(f"Ψ(x, τ)  [{u}" + "$^{-1/2}$]")
        ax1.set_title(f"Función de onda en τ = {num(tau, 4)}")
        ax1.legend(loc="upper right")
        ax2.plot(xs, rho * f, color=COLOR["densidad"], lw=2.4, label="|Ψ(x, τ)|²")
        ax2.fill_between(xs, rho * f, where=(xf >= u1) & (xf <= u2), color=COLOR["verde"], alpha=0.25,
                         label=f"región [{num(u1, 3)}a, {num(u2, 3)}a]")
        if individuales:
            for n, c in zip(ns, cs):
                ax2.plot(xs, abs(c) ** 2 * (2 / a_m) * np.sin(n * np.pi * xf) ** 2 * f, lw=1.2, ls=":",
                         label=f"|c{int(n)}|²·|ψ{int(n)}|²")
        _paredes(ax2, 0, a_m / f)
        ax2.set_xlabel(f"posición x [{u}]")
        ax2.set_ylabel(f"|Ψ|²  [{u}$^{{-1}}$]")
        ax2.set_title("Densidad de probabilidad")
        ax2.legend(loc="upper right", fontsize=8.5)
        P = valor_esperado(ns, cs, S, tau)
        recuadro(ax2, f"P(región) = {num(P, 5)}\n⟨x⟩ = {num(valor_esperado(ns, cs, X, tau), 5)}·a",
                 x=0.03, y=0.95, ha="left", color=COLOR["verde"])
        fila_inf = 1
    else:
        fila_inf = 0
    if mapa and representable:
        ax3 = fig.add_subplot(gs[fila_inf, 0])
        xm = np.linspace(0, 1, max(301, 12 * int(ns.max())))
        tm = np.linspace(0, 2 * np.pi, 181)
        M = np.array([np.abs(onda_total(xm, ns, cs, t, a_m)) ** 2 * a_m for t in tm])
        ax3.imshow(M, origin="lower", aspect="auto", extent=[0, a_m / f, 0, 2 * np.pi], cmap="magma")
        ax3.axhline(math.remainder(tau, 2 * math.pi) % (2 * math.pi), color="white", lw=1.5, ls="--")
        ax3.set_xlabel(f"posición x [{u}]")
        ax3.set_ylabel("tiempo τ = E₁t/ħ")
        ax3.set_title("Mapa espacio–tiempo de |Ψ|² (un periodo)")
        ax3.grid(False)
        ax4 = fig.add_subplot(gs[fila_inf, 1])
    else:
        ax4 = fig.add_subplot(gs[fila_inf, :])
    ax4.plot(taus, x_t, color=COLOR["onda"], lw=2.3, label="⟨x⟩/a")
    ax4.plot(taus, P_t, color=COLOR["verde"], lw=2.3, label="P(región)")
    ax4.axvline(math.remainder(tau, 2 * math.pi) % (2 * math.pi), color=COLOR["rojo"], ls="--", lw=1.4,
                label="τ elegido")
    ax4.set_xlim(0, 2 * np.pi)
    ax4.set_ylim(-0.02, 1.02)
    ax4.set_xlabel("tiempo τ = E₁t/ħ (un periodo completo)")
    ax4.set_ylabel("valor")
    ax4.set_title("Evolución temporal (fórmulas exactas)")
    ax4.legend(loc="upper right")
    fig.suptitle("QUANTUM LAB — Partícula en una caja 1D · superposición de estados", fontsize=16, fontweight="bold")
    return fig


In [ ]:
#@title 4) Resultados en texto (mismo formato que los simuladores de Bohr y del efecto fotoeléctrico) { display-mode: "form" }
def informe_estado(n_valor, a_nm, m_me, diag=None, dibujar=True):
    """Imprime los resultados de un estado estacionario y dibuja sus gráficas."""
    diag = diag or Diagnostico()
    titulo_caja("QUANTUM LAB — PARTÍCULA EN UNA CAJA 1D · ESTADO ESTACIONARIO")
    n, _ = revisar_n(n_valor, diag)
    a_m, m_kg = revisar_caja(a_nm, m_me, diag)

    seccion("PARÁMETROS DEL SISTEMA")
    dato("Longitud de la caja a", num(a_nm), "nm" + (f"   = {num(a_m)} m" if a_m else ""))
    dato("Masa de la partícula m", num(m_me), "mₑ" + (f"   = {num(m_kg)} kg" if m_kg else ""))
    dato("Número cuántico n", entero(n) if n is not None else num(n_valor))
    if diag.hay_error or n is None or a_m is None or m_kg is None:
        diag.imprimir()
        return None

    logE = log10_energia(n, a_m, m_kg)
    log_e1 = log10_energia(1, a_m, m_kg)
    revisar_relatividad(logE, m_kg, diag, f"La energía del nivel n = {entero(n)}")
    lam = 2 * a_m / n
    if lam < L_PLANCK:
        diag.aviso(f"La longitud de onda de De Broglie (2a/n = {num(lam)} m) es menor que la longitud de Planck: "
                   "ninguna teoría conocida describe ondas tan cortas.")
    log_sep_ev = math.log10(2 * n + 1) + log_e1 - math.log10(EV)
    if log_sep_ev < math.log10(KT_AMBIENTE_EV):
        diag.nota(f"La separación con el nivel siguiente ({num_log(log_sep_ev)} eV) es menor que la energía térmica a "
                  "temperatura ambiente (0.026 eV): a 300 K la partícula ocuparía muchísimos niveles a la vez y se "
                  "comportaría de forma casi clásica.")
    if n > 100:
        diag.nota(f"Con n = {entero(n)} la densidad |ψ|² oscila tan deprisa que su promedio en cualquier tramo es "
                  "uniforme, 1/a: es la probabilidad clásica de una partícula que rebota entre las paredes "
                  "(principio de correspondencia).")
    if n > N_DIBUJO_MAX:
        diag.nota(f"Dibujar ψ en toda la caja exigiría más de {entero(40 * n)} puntos. La gráfica muestra una ventana "
                  "junto a la pared con las 3 primeras semilongitudes de onda; los valores numéricos son exactos.")
    diag.imprimir()

    seccion("RESULTADOS DEL ESTADO")
    dato("Energía Eₙ = n²π²ħ²/(2ma²)", num_log(logE - math.log10(EV)), "eV")
    dato("Energía Eₙ", num_log(logE), "J")
    dato("Energía Eₙ (unidades atómicas)", num_log(logE - math.log10(E_H)), "Eₕ")
    dato("Energía del fundamental E₁", num_log(log_e1 - math.log10(EV)), "eV")
    dato("Cociente Eₙ / E₁ = n²", num_log(2 * math.log10(n)))
    dato("Separación Eₙ₊₁ − Eₙ = (2n+1)·E₁", num_log(log_sep_ev), "eV")
    log_p = math.log10(n) + math.log10(math.pi * HBAR) - math.log10(a_m)
    dato("Momento |p| = nπħ/a", num_log(log_p), "kg·m/s")
    log_v = log_p - math.log10(m_kg)
    dato("Velocidad equivalente v = |p|/m", num_log(log_v), "m/s" + f"   ({num_log(log_v - math.log10(C_LUZ), 3)} c)")
    dato("Longitud de onda de De Broglie λ = 2a/n", num(lam), "m" + f"   = {num(lam * 1e9)} nm")
    dato("Nodos interiores (n − 1)", entero(n - 1))
    if n <= 8:
        f, u = escala_longitud(a_m)
        dato("Posición de los nodos", (", ".join(num(k * a_m / n / f, 4) for k in range(1, n)) + f" {u}") if n > 1
             else "ninguno")
        dato("Máximos de probabilidad", ", ".join(num((2 * k - 1) * a_m / (2 * n) / f, 4) for k in range(1, n + 1)), u)
    dato("Valor medio ⟨x⟩", "a/2 =", f"{num(a_m / 2)} m")
    dato("Incertidumbre Δx = a√(1/12 − 1/2n²π²)", num(incertidumbre_posicion(n, 1.0), 5), "· a")
    if n <= N_DIBUJO_MAX:
        x = np.linspace(0, a_m, max(4001, 40 * n + 1))
        dato("Normalización numérica ∫|ψ|²dx", f"{integrar(x, funcion_onda(x, n, a_m) ** 2):.8f}", "(debe ser 1)")

    seccion("NIVELES DE ENERGÍA CERCANOS")
    ks = list(range(max(1, n - 2), n + 3)) if n > 3 else list(range(1, 6))
    tabla(["n", "Eₙ [eV]", "Eₙ / E₁", "λ = 2a/n [nm]"],
          [[("» " if k == n else "") + entero(k), num_log(log10_energia(k, a_m, m_kg) - math.log10(EV)),
            num_log(2 * math.log10(k)), num(2 * a_nm / k)] for k in ks])

    seccion("INTERPRETACIÓN FÍSICA")
    vinetas([
        "Solo existen las energías Eₙ = n²·E₁: la condición ψ = 0 en las paredes cuantiza la energía.",
        f"E₁ ≠ 0 (energía de punto cero): confinar la partícula en una longitud a obliga a un momento mínimo ~ħ/a.",
        "Eₙ ∝ 1/(m·a²): partículas más ligeras o cajas más pequeñas dan niveles más separados (efectos cuánticos "
        "más visibles).",
        f"La función de onda tiene n − 1 = {entero(n - 1)} nodos: puntos donde nunca se encuentra la partícula.",
        "|ψ|² es una densidad de probabilidad: al medir la posición muchas veces, los resultados se reparten según ella.",
    ])
    if dibujar:
        mostrar(figura_estado(n, a_m, m_kg, logE))
    return {"n": n, "logE": logE, "lambda": lam}


def informe_superposicion(estados, tau, u1, u2, a_nm, m_me, individuales=False, diag=None, dibujar=True):
    """Imprime y dibuja una superposición. estados = [(n, amplitud, fase), …] con los estados activos."""
    diag = diag or Diagnostico()
    titulo_caja("QUANTUM LAB — PARTÍCULA EN UNA CAJA 1D · SUPERPOSICIÓN")
    a_m, m_kg = revisar_caja(a_nm, m_me, diag)
    if not estados:
        diag.error("No hay ningún estado activo. Marca al menos una casilla «activo».")
    ns, cs = coeficientes(estados, diag)
    if tau is not None and abs(tau) > 2 * math.pi:
        diag.nota(f"τ = {num(tau)} supera un periodo. Como cada término gira con fase n²τ, la evolución se repite "
                  f"exactamente cada Δτ = 2π: se usa τ = {num(tau % (2 * math.pi))}.")
    if tau is not None and tau < 0:
        diag.nota("τ negativo: se calcula el estado en un instante ANTERIOR (la ecuación de Schrödinger es reversible).")
    if u1 is not None and u2 is not None:
        if u1 > u2:
            diag.nota("El inicio de la región es mayor que el final: se intercambian.")
            u1, u2 = u2, u1
        if u1 < 0 or u2 > 1:
            diag.nota("La región sale de la caja. Fuera de las paredes la probabilidad es exactamente 0 (potencial "
                      "infinito): solo cuenta la parte entre 0 y a.")
            u1, u2 = min(max(u1, 0.0), 1.0), min(max(u2, 0.0), 1.0)
        if u1 == u2:
            diag.nota("La región tiene anchura cero: la probabilidad de encontrar la partícula en un punto exacto es 0.")
    seccion("PARÁMETROS DEL SISTEMA")
    dato("Longitud de la caja a", num(a_nm), "nm")
    dato("Masa de la partícula m", num(m_me), "mₑ")
    dato("Tiempo τ = E₁t/ħ", num(tau))
    dato("Región estudiada", f"[{num(u1)}·a, {num(u2)}·a]")
    if diag.hay_error or ns is None or a_m is None or m_kg is None or tau is None or u1 is None or u2 is None:
        diag.imprimir()
        return None
    log_e1 = log10_energia(1, a_m, m_kg)
    revisar_relatividad(log_e1 + 2 * math.log10(ns.max()), m_kg, diag, f"La energía del nivel más alto (n = {entero(ns.max())})")
    if ns.max() ** 2 * abs(tau % (2 * math.pi)) > 1e14:
        diag.aviso("Las fases n²τ superan 1e14 rad: el ordenador ya no puede seguirlas con precisión y la dependencia "
                   "temporal fina se vuelve numéricamente aleatoria (los promedios siguen siendo correctos).")
    if ns.max() > N_DIBUJO_MAX:
        diag.nota(f"Con n = {entero(ns.max())} la densidad no se puede dibujar en toda la caja. Se muestran las magnitudes "
                  "exactas (fórmulas cerradas) y su evolución temporal.")
    diag.imprimir()

    tau_r = tau % (2 * math.pi)
    S = matriz_region(ns, u1, u2)
    X = matriz_posicion(ns)
    e_k = ns ** 2
    pesos = np.abs(cs) ** 2
    e_media = float(np.sum(pesos * e_k))
    delta_e = math.sqrt(max(float(np.sum(pesos * e_k ** 2)) - e_media ** 2, 0.0))
    e1_ev = 10 ** (log_e1 - math.log10(EV)) if log_e1 < 300 else math.inf

    seccion("COEFICIENTES NORMALIZADOS")
    tabla(["n", "cₙ (normalizado)", "|cₙ|²", "fase [rad]", "Eₙ [eV]"],
          [[entero(n), f"{c.real:+.4f} {c.imag:+.4f}i", f"{abs(c) ** 2:.6f}", f"{math.atan2(c.imag, c.real):+.4f}",
            num_log(log_e1 + 2 * math.log10(n) - math.log10(EV))] for n, c in zip(ns, cs)])
    seccion("RESULTADOS EN EL INSTANTE ELEGIDO")
    dato("Suma Σ|cₙ|² (normalización)", f"{float(np.sum(pesos)):.10f}")
    dato("Tiempo real t = τħ/E₁", num(tau * HBAR / (e1_ev * EV)) if math.isfinite(e1_ev) else "≈ 0", "s")
    dato("Periodo de la evolución T = 2πħ/E₁", num(2 * math.pi * HBAR / (e1_ev * EV)) if math.isfinite(e1_ev) else "≈ 0", "s")
    dato("Probabilidad en la región P", f"{valor_esperado(ns, cs, S, tau_r):.6f}")
    dato("Probabilidad clásica (uniforme)", f"{u2 - u1:.6f}", "(anchura de la región / a)")
    dato("Posición media ⟨x⟩", f"{valor_esperado(ns, cs, X, tau_r):.6f}", "· a")
    dato("Energía media ⟨E⟩ = Σ|cₙ|²Eₙ", num(e_media * e1_ev), "eV")
    dato("Dispersión de la energía ΔE", num(delta_e * e1_ev), "eV")
    seccion("INTERPRETACIÓN FÍSICA")
    lineas = ["Cada término gira con su propia frecuencia Eₙ/ħ. Lo que se observa (|Ψ|²) depende de las fases "
              "RELATIVAS: los términos cruzados 2·Re[c*ⱼcₖψⱼψₖe^{−i(Eₖ−Eⱼ)t/ħ}] producen la interferencia.",
              "Si solo hay un estado, |Ψ|² no cambia con el tiempo (estado estacionario): la fase global no se mide.",
              "Las probabilidades |cₙ|² de obtener cada energía NO cambian con el tiempo: ⟨E⟩ y ΔE son constantes.",
              "⟨x⟩ oscila a las frecuencias (Eₖ − Eⱼ)/ħ: la partícula «rebota» entre las paredes."]
    if len(ns) == 1:
        lineas.insert(0, "Solo hay un estado activo: es un estado estacionario y la densidad no evoluciona.")
    vinetas(lineas)
    if dibujar:
        mostrar(figura_superposicion(ns, cs, tau_r, a_m, u1, u2, individuales))
    return {"ns": ns, "cs": cs, "P": valor_esperado(ns, cs, S, tau_r), "x": valor_esperado(ns, cs, X, tau_r)}


In [ ]:
#@title 5) Pruebas numéricas (comprueban que los cálculos son correctos) { display-mode: "form" }
def ejecutar_pruebas():
    import io, contextlib
    resultados = []

    def prueba(nombre, condicion, detalle=""):
        resultados.append((nombre, bool(condicion), detalle))

    def silencioso(funcion, *args, **kwargs):
        with contextlib.redirect_stdout(io.StringIO()):
            return funcion(*args, **kwargs)

    a, m = 1e-9, M_E
    x = np.linspace(0, a, 20001)
    for n in (1, 7):
        prueba(f"Normalización ∫|ψ{n}|²dx = 1", abs(integrar(x, funcion_onda(x, n, a) ** 2) - 1) < 1e-6)
    prueba("Ortogonalidad ∫ψ₁ψ₂dx = 0", abs(integrar(x, funcion_onda(x, 1, a) * funcion_onda(x, 2, a))) < 1e-8)
    e1 = energia_nivel(1, a, m) / EV
    prueba("Electrón en 1 nm: E₁ ≈ 0.376 eV", abs(e1 - 0.37603) < 1e-4, f"{e1:.5f} eV")
    prueba("Eₙ/E₁ = n² (n = 5)", abs(energia_nivel(5, a, m) / energia_nivel(1, a, m) - 25) < 1e-12)
    prueba("log₁₀ E coincide con la fórmula directa", abs(10 ** log10_energia(3, a, m) / energia_nivel(3, a, m) - 1) < 1e-12)
    psi4 = funcion_onda(x, 4, a)
    prueba("ψ₄ tiene 3 nodos interiores", int(np.sum(np.diff(np.sign(psi4[1:-1])) != 0)) == 3)
    rho3 = funcion_onda(x, 3, a) ** 2
    dx_num = math.sqrt(integrar(x, x ** 2 * rho3) - integrar(x, x * rho3) ** 2)
    prueba("Δx numérico = a√(1/12 − 1/2n²π²)", abs(dx_num / incertidumbre_posicion(3, a) - 1) < 1e-6, f"{dx_num / a:.6f} a")
    ns, cs = np.array([1.0, 2.0, 3.0]), np.array([0.6, 0.48 + 0.36j, 0.52j])
    cs = cs / np.sqrt(np.sum(np.abs(cs) ** 2))
    xf = np.linspace(0, 1, 40001)
    for tau in (0.0, 1.3):
        rho = np.abs(onda_total(xf, ns, cs, tau, 1.0)) ** 2
        x_num = integrar(xf, xf * rho)
        x_ana = valor_esperado(ns, cs, matriz_posicion(ns), tau)
        prueba(f"⟨x⟩ exacto = numérico (τ = {tau})", abs(x_num - x_ana) < 1e-7, f"{x_ana:.7f}")
        xr = np.linspace(0.2, 0.7, 40001)
        P_num = integrar(xr, np.abs(onda_total(xr, ns, cs, tau, 1.0)) ** 2)
        P_ana = valor_esperado(ns, cs, matriz_region(ns, 0.2, 0.7), tau)
        prueba(f"P(región) exacta = numérica (τ = {tau})", abs(P_num - P_ana) < 1e-6, f"{P_ana:.7f}")
    prueba("Periodicidad: Ψ(τ + 2π) = Ψ(τ)", np.allclose(onda_total(xf[::50], ns, cs, 0.7, 1.0),
                                                       onda_total(xf[::50], ns, cs, 0.7 + 2 * math.pi, 1.0)))
    prueba("P en toda la caja = 1", abs(valor_esperado(ns, cs, matriz_region(ns, 0, 1), 0.4) - 1) < 1e-12)
    # Datos no físicos o extremos: deben explicarse, no romper el programa
    for etiqueta, args in [("n = 2.5", (2.5, 1, 1)), ("n = 0", (0, 1, 1)), ("masa 0", (1, 1, 0)), ("masa negativa", (1, 1, -1)),
                           ("longitud negativa", (1, -1, 1))]:
        d = Diagnostico()
        r = silencioso(informe_estado, *args, diag=d, dibujar=False)
        prueba(f"Dato no físico ({etiqueta}): se explica y no se calcula", r is None and d.hay_error)
    d = Diagnostico()
    r = silencioso(informe_estado, -3, 1, 1, diag=d, dibujar=False)
    prueba("n = −3 se trata como n = 3 (mismo estado)", r is not None and r["n"] == 3)
    d = Diagnostico()
    r = silencioso(informe_estado, 1e12, 1, 1, diag=d, dibujar=False)
    prueba("n = 10¹² se calcula sin desbordar y avisa de la relatividad",
           r is not None and any(n_ == "AVISO" for n_, _ in d.items))
    d = Diagnostico()
    silencioso(informe_estado, 1, 1e-30, 1, diag=d, dibujar=False)
    prueba("a menor que la longitud de Planck: se avisa", any("Planck" in t for _, t in d.items))
    prueba("Lectura de «2,5», «3*10^8» y «2*pi»", leer_numero("2,5")[0] == 2.5 and leer_numero("3*10^8")[0] == 3e8
           and abs(leer_numero("2*pi")[0] - 2 * math.pi) < 1e-15)
    prueba("Texto no numérico («abc») se explica", leer_numero("abc")[0] is None)

    titulo_caja("PRUEBAS NUMÉRICAS — PARTÍCULA EN UNA CAJA")
    tabla(["#", "Comprobación", "Resultado", "Detalle"], alinear="clcl", filas=
          [[i, nombre, "Correcta" if ok else "FALLA", detalle] for i, (nombre, ok, detalle) in enumerate(resultados, 1)],
          sangria=1)
    correctas = sum(ok for _, ok, _ in resultados)
    print(f"\n   {correctas} de {len(resultados)} pruebas superadas.")
    return correctas, len(resultados)


In [ ]:
#@title 6) Simulador interactivo — partícula en una caja 1D { display-mode: "form" }
EJEMPLOS = {
    "Electrón en una caja de 1 nm (n = 1)": {"a": "1", "m": "1", "n": "1"},
    "Electrón en una caja de 1 nm (n = 3)": {"a": "1", "m": "1", "n": "3"},
    "Electrón confinado en un átomo (a = 0.1 nm)": {"a": "0.1", "m": "1", "n": "1"},
    "Protón en un núcleo (a = 1e-5 nm)": {"a": "1e-5", "m": "1836.15", "n": "1"},
    "Pelota de 0.145 kg en 1 m (límite clásico)": {"a": "1e9", "m": "1.592e29", "n": "1"},
    "Valor extremo: n = 10^9": {"a": "1", "m": "1", "n": "1e9"},
    "Valor no físico: n = 2.5": {"a": "1", "m": "1", "n": "2.5"},
    "Valor no físico: masa negativa": {"a": "1", "m": "-1", "n": "1"},
}

entrada_a = Entrada("Longitud de la caja a", 1.0, "nm", tipo="log", rango=(-2, 3),
                    ayuda="Anchura del pozo de potencial infinito")
entrada_m = Entrada("Masa de la partícula m", 1.0, "mₑ (masas del electrón)", tipo="log", rango=(-1, 4))
entrada_n = Entrada("Número cuántico n", 1, "", tipo="entero", rango=(1, 20))
menu_ejemplo, fila_ejemplo = selector("Configuración de ejemplo", ["— elegir —"] + list(EJEMPLOS), "— elegir —", "420px")

# Superposición: el usuario elige cuántos estados (filas) y cuáles; para listas largas hay un campo de texto sin límite
import re
N_FILAS_MAX = 30
_cab = w.HBox([w.HTML("<b>Estado</b>", layout=w.Layout(width="80px")), w.HTML("<b>activo</b>", layout=w.Layout(width="70px")),
               w.HTML("<b>n</b>", layout=w.Layout(width="150px")), w.HTML("<b>coeficiente (amplitud A)</b>", layout=w.Layout(width="150px")),
               w.HTML("<b>fase φ [rad]</b>", layout=w.Layout(width="150px"))])
filas_sup = []                      # cada fila: (activo, n, amplitud, fase)
_cajas_fila = []
caja_estados = w.VBox([_cab])
entrada_nest = Entrada("Número de estados a superponer", 2, "(filas de la tabla)", tipo="entero", rango=(1, 10))
lista_rapida = w.Textarea(value="", continuous_update=False, rows=2,
                          placeholder="Opcional, sin límite de estados.  Formato  n:coeficiente[:fase]  separados por «;» o por líneas.  "
                                      "Ejemplo:  1:1; 2:0.5; 7:-0.3; 12:0.2:1.57",
                          layout=w.Layout(width="760px"))


def _crear_fila(i, n0, A0, f0, act):
    fila = (w.Checkbox(value=act, indent=False, layout=w.Layout(width="70px")),
            w.Text(value=n0, continuous_update=False, layout=w.Layout(width="140px")),
            w.Text(value=A0, continuous_update=False, layout=w.Layout(width="140px")),
            w.Text(value=f0, continuous_update=False, layout=w.Layout(width="140px")))
    for control in fila:
        control.observe(lambda _: calcular_superposicion(), "value")
    caja = w.HBox([w.HTML(f"{i}", layout=w.Layout(width="80px"))] + list(fila), layout=w.Layout(margin="1px 0"))
    return fila, caja


def _reconstruir_filas(N):
    """Deja exactamente N filas (conserva las existentes; las nuevas empiezan en n = fila, A = 1, φ = 0)."""
    del filas_sup[N:]
    del _cajas_fila[N:]
    for i in range(len(filas_sup) + 1, N + 1):
        fila, caja = _crear_fila(i, str(i), "1", "0", True)
        filas_sup.append(fila)
        _cajas_fila.append(caja)
    caja_estados.children = [_cab] + list(_cajas_fila)


_reconstruir_filas(2)
entrada_tau = Entrada("Tiempo τ = E₁t/ħ", 0.0, "(periodo 2π)", tipo="real", rango=(0, 2 * math.pi), paso=0.05)
entrada_u1 = Entrada("Región: inicio", 0.25, "· a", tipo="real", rango=(0, 1), paso=0.01)
entrada_u2 = Entrada("Región: final", 0.75, "· a", tipo="real", rango=(0, 1), paso=0.01)
casilla_ind = casilla("Mostrar las densidades de cada estado", False)

salida_estado, salida_sup, salida_pruebas = w.Output(), w.Output(), w.Output()


def _leer_texto(caja_texto, etiqueta, diag):
    return _leer_valor(caja_texto.value, etiqueta, diag)


def _leer_valor(texto, etiqueta, diag):
    v, problema = leer_numero(texto)
    if problema:
        diag.error(f"{etiqueta}: {problema}.")
        return None
    if not math.isfinite(v):
        diag.error(f"{etiqueta} es infinito: escribe un valor finito.")
        return None
    return v


def calcular_estado():
    def tarea():
        d = Diagnostico()
        informe_estado(entrada_n.leer(d), entrada_a.leer(d), entrada_m.leer(d), d)
    ejecutar_en(salida_estado, tarea)


def _estados_de_lista(texto, diag):
    estados = []
    for k, item in enumerate((t.strip() for t in re.split(r"[;\n]+", texto)), 1):
        if not item:
            continue
        partes = [p.strip() for p in item.split(":")]
        if len(partes) > 3 or not partes[0]:
            diag.error(f"Elemento {k} de la lista («{item}»): usa el formato n:coeficiente  o  n:coeficiente:fase.")
            continue
        partes += ["1", "0"][len(partes) - 1:]
        estados.append((_leer_valor(partes[0], f"n del elemento {k}", diag),
                        _leer_valor(partes[1] or "1", f"Coeficiente del elemento {k}", diag),
                        _leer_valor(partes[2] or "0", f"Fase del elemento {k}", diag)))
    return estados


def _cambiar_numero_estados():
    d = Diagnostico()
    N = entrada_nest.leer(d)
    if N is not None:
        _reconstruir_filas(int(min(max(N, 1), N_FILAS_MAX)))
    calcular_superposicion()


def calcular_superposicion():
    def tarea():
        d = Diagnostico()
        estados = []
        N = entrada_nest.leer(d)
        if N is not None and N > N_FILAS_MAX:
            d.nota(f"La tabla admite {N_FILAS_MAX} filas para que el navegador siga fluido. Para superponer {entero(N)} estados "
                   "usa la «Lista rápida» (no tiene límite de estados).")
        if lista_rapida.value.strip():
            d.nota("Se usa la «Lista rápida»; las filas de la tabla se ignoran mientras la lista no esté vacía.")
            estados = _estados_de_lista(lista_rapida.value, d)
        else:
            for i, (act, n, A, fa) in enumerate(filas_sup, 1):
                if act.value:
                    estados.append((_leer_texto(n, f"n del estado {i}", d), _leer_texto(A, f"Amplitud del estado {i}", d),
                                    _leer_texto(fa, f"Fase del estado {i}", d)))
        informe_superposicion(estados, entrada_tau.leer(d), entrada_u1.leer(d), entrada_u2.leer(d),
                              entrada_a.leer(d), entrada_m.leer(d), casilla_ind.value, d)
    ejecutar_en(salida_sup, tarea)


def calcular_todo():
    calcular_estado()
    calcular_superposicion()


def _aplicar_ejemplo(cambio):
    e = EJEMPLOS.get(cambio["new"])
    if not e:
        return
    entrada_a.texto.value, entrada_m.texto.value, entrada_n.texto.value = e["a"], e["m"], e["n"]


menu_ejemplo.observe(_aplicar_ejemplo, "value")
for e in (entrada_a, entrada_m):
    e.al_cambiar(calcular_todo)
entrada_n.al_cambiar(calcular_estado)
for e in (entrada_tau, entrada_u1, entrada_u2):
    e.al_cambiar(calcular_superposicion)
casilla_ind.observe(lambda _: calcular_superposicion(), "value")
entrada_nest.al_cambiar(_cambiar_numero_estados)
lista_rapida.observe(lambda _: calcular_superposicion(), "value")

pestanas = w.Tab(children=[
    w.VBox([panel("Estado estacionario", [entrada_n.caja]), salida_estado]),
    w.VBox([panel("Estados de la superposición (los coeficientes se normalizan automáticamente)", [entrada_nest.caja, caja_estados, w.HTML("<div style='margin:6px 0 2px'>Lista rápida (opcional)</div>"), lista_rapida]),
            panel("Tiempo y región de probabilidad", [entrada_tau.caja, entrada_u1.caja, entrada_u2.caja, casilla_ind]),
            salida_sup]),
    w.VBox([boton("Ejecutar las pruebas numéricas", lambda: ejecutar_en(salida_pruebas, ejecutar_pruebas)), salida_pruebas]),
])
for i, t in enumerate(["Estado individual", "Superposición", "Pruebas numéricas"]):
    pestanas.set_title(i, t)

encabezado("Partícula en una caja 1D",
           "Pozo de potencial infinito: estados ψₙ, energías Eₙ ∝ n², longitud de onda de De Broglie, nodos y "
           "superposición dependiente del tiempo.")
display(w.VBox([panel("Parámetros del sistema", [ayuda_entradas(), fila_ejemplo, entrada_a.caja, entrada_m.caja]),
                pestanas]))
calcular_todo()
